# CytoBridge tutorial: ARISTA salamander brain regeneration

This notebook is a compact, public walkthrough for readers who already have an
aligned CytoBridge AnnData file and, normally, a trained checkpoint. It keeps the
scientific workflow visible while delegating numerical work to the package API.

**Prerequisites:** Python, Jupyter, `CytoBridge[all]`, an aligned `.h5ad`, a trained
model directory, a ligand-receptor table, and a CUDA device for practical full runs.
The formal workflow and training presets are read from the installed wheel.

**Learning goals:** load and check the data contract; inspect the packaged dataset preset;
optionally train; load a checkpoint; interpolate; and calculate composition, velocity,
growth, communication, ligand-receptor trajectories, and unwarped distribution metrics.

The notebook is distributed with empty outputs. Values shown after execution come from
the paths you provide; no manuscript result is embedded or invented here. By default it
runs a compact walkthrough. Set `RUN_FORMAL_SCOPE=True` to use the full time grid and
population cap from the packaged preset; both modes use the same scientific solver settings.


## Outline

1. Set the external data paths.
2. Load the aligned AnnData and wheel-packaged formal preset.
3. Optionally train, or load an existing checkpoint.
4. Interpolate and classify cell states.
5. Run downstream summaries on the unwarped model state.
6. Interpret results, review pitfalls, and try controlled exercises.


In [ ]:
from __future__ import annotations

from importlib import resources
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import torch
import yaml

import CytoBridge as cb
from CytoBridge.workflow import load_workflow_config

DATASET_PRESET = 'arista'
workflow_preset, workflow_preset_source = load_workflow_config(DATASET_PRESET)
dataset_preset = workflow_preset["dataset"]
scientific_preset = workflow_preset["scientific"]
training_preset = workflow_preset["train"]
downstream_preset = workflow_preset["downstream"]

SEED = int(scientific_preset["seed"])
ALPHA_EXPRESS = float(scientific_preset["alpha_express"])
ALPHA_SPATIAL = float(scientific_preset["alpha_spatial"])
K_NEIGHBORS = int(scientific_preset["classifier_k"])
INTERACTION_CUTOFF = float(training_preset["interaction_cutoff"])
EDGE_PREDICTOR_THRESHOLD = float(training_preset["edge_predictor_threshold"])
CLASSIFIER_EPOCHS = int(downstream_preset["classifier_epochs"])
CLASSIFIER_HIDDEN_SIZE = int(downstream_preset["classifier_hidden_size"])
CLASSIFIER_LR = float(downstream_preset["classifier_lr"])
CLASSIFIER_BEST_METRIC = str(downstream_preset["classifier_best_metric"])
CLASSIFIER_STRICT_STRATIFICATION = bool(
    downstream_preset["classifier_strict_stratification"]
)
FORMAL_OBSERVED_TIMES = [float(t) for t in downstream_preset["observed"]]
FORMAL_INTERPOLATED_TIMES = [
    float(t) for t in downstream_preset["interpolated"]
]
FORMAL_SDE_N_SAMPLES = downstream_preset["sde_n_samples"]
SDE_DT = float(downstream_preset["sde_dt"])
SPLIT_SDE_DT = float(downstream_preset["split_sde_dt"])
SPLIT_SIGMA = float(downstream_preset["split_sigma"])
SPLIT_GROWTH_ALPHA = float(downstream_preset["split_growth_alpha"])
LINEAGE_ENABLED = bool(downstream_preset.get("lineage_enabled", False))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# False keeps the notebook quick by using midpoint interpolation and at most
# 5,000 generated particles. Observed-slice analyses still use the supplied AnnData.
# by the formal workflow preset. Solver settings never change between modes.
RUN_FORMAL_SCOPE = False
COMPACT_PARTICLES = 5_000


## 1. Inputs

Set the aligned AnnData, trained model, and ligand-receptor paths below. No source checkout or external training YAML is needed: the notebook loads the formal `arista` workflow and training presets from the installed wheel. Set `EDGE_PREDICTOR_PATH` only when deliberately fitting a new model.

The available ARISTA source contains integer counts in layers['counts'] and an already transformed X matrix. The canonical adapter uses the counts layer once. Because the source is already restricted to 2,000 genes, describe this as a fixed-2,000-gene clean rerun.


In [ ]:
ALIGNED_H5AD = Path("inputs/arista_aligned.h5ad")
MODEL_DIR = Path("inputs/arista_model")
LR_DATABASE = Path("inputs/CellChatDB.ligrec.human.csv")
OUTPUT_DIR = Path("tutorial_outputs/arista")

# A new training run needs the dataset-specific edge-classifier checkpoint.
# Existing trained models do not need this path here.
EDGE_PREDICTOR_PATH: Path | None = None
RUN_TRAINING = False

required_inputs = {
    "aligned_h5ad": ALIGNED_H5AD,
    "lr_database": LR_DATABASE,
    **(
        {"edge_predictor_path": EDGE_PREDICTOR_PATH}
        if RUN_TRAINING
        else {"model_dir": MODEL_DIR}
    ),
}
missing_inputs = [
    name
    for name, path in required_inputs.items()
    if path is None or not Path(path).exists()
]
if missing_inputs:
    missing_lines = "\n".join(
        f"  - {name}: {required_inputs[name]}" for name in missing_inputs
    )
    raise FileNotFoundError(
        f"Provide the external files for the wheel-packaged '{DATASET_PRESET}' preset.\n"
        f"Missing required input(s):\n{missing_lines}\n"
        "RUN_TRAINING=False expects an existing model directory; set it to True only "
        "for a deliberate new fit and provide EDGE_PREDICTOR_PATH."
    )

pd.Series(
    {
        "workflow_preset": workflow_preset_source,
        "aligned_h5ad": ALIGNED_H5AD,
        "model_dir": MODEL_DIR,
        "lr_database": LR_DATABASE,
        "device": DEVICE,
        "run_training": RUN_TRAINING,
    }
)


### Dataset time and expression contract

Use obs['time_point_processed'] as the model time axis. Keep the biological time labels in a separate observation column for figures rather than passing strings into the dynamical model.

The available ARISTA source contains integer counts in layers['counts'] and an already transformed X matrix. The canonical adapter uses the counts layer once. Because the source is already restricted to 2,000 genes, describe this as a fixed-2,000-gene clean rerun.


In [ ]:
adata = sc.read_h5ad(ALIGNED_H5AD)
TIME_KEY = str(dataset_preset["time_key"])
ANNOTATION_KEY = str(dataset_preset["annotation_key"])
LATENT_KEY = str(dataset_preset["obsm_key"])
SPATIAL_KEY = str(dataset_preset["spatial_key"])
CONCAT_SPATIAL = bool(dataset_preset.get("concat_spatial", True))

required_obs = {TIME_KEY, ANNOTATION_KEY}
required_obsm = {LATENT_KEY, SPATIAL_KEY}
assert required_obs.issubset(adata.obs.columns)
assert required_obsm.issubset(adata.obsm)

available_observed_times = sorted(
    pd.to_numeric(adata.obs[TIME_KEY], errors="raise").unique().astype(float)
)
missing_formal_times = [
    time
    for time in FORMAL_OBSERVED_TIMES
    if not any(np.isclose(time, value) for value in available_observed_times)
]
if missing_formal_times:
    raise ValueError(f"Aligned input is missing formal time anchors: {missing_formal_times}")
observed_times = list(FORMAL_OBSERVED_TIMES)
input_summary = pd.Series(
    {
        "cells": adata.n_obs,
        "genes": adata.n_vars,
        "available_model_times": available_observed_times,
        "formal_observed_times": observed_times,
        "spatial_dimensions": adata.obsm[SPATIAL_KEY].shape[1],
        "latent_dimensions": adata.obsm[LATENT_KEY].shape[1],
        "cell_types": adata.obs[ANNOTATION_KEY].astype(str).nunique(),
    }
)
input_summary


## 2. Resolve the wheel-packaged formal preset

The dataset schema, scientific constants, formal graph cutoff, and edge-classifier threshold come from the installed workflow preset. The complete stage plan comes from its packaged training config. These values remain dataset-specific.


In [ ]:
training_config_name = str(training_preset["config"])
training_config_resource = (
    resources.files("CytoBridge")
    .joinpath("configs")
    .joinpath(training_config_name)
)
if not training_config_resource.is_file():
    raise FileNotFoundError(
        f"Installed CytoBridge wheel is missing {training_config_name!r}."
    )

config = yaml.safe_load(training_config_resource.read_text(encoding="utf-8"))
config["seed"] = SEED
config["ckpt_dir"] = str(MODEL_DIR)
config["training"]["defaults"]["alpha_express"] = ALPHA_EXPRESS
config["training"]["defaults"]["alpha_spatial"] = ALPHA_SPATIAL

interaction_config = config["model"]["interaction_net"]
interaction_config["cutoff"] = INTERACTION_CUTOFF
interaction_config["edge_predictor_thre"] = EDGE_PREDICTOR_THRESHOLD
interaction_config["edge_predictor_path"] = (
    None if EDGE_PREDICTOR_PATH is None else str(EDGE_PREDICTOR_PATH)
)

resolved_settings = pd.Series(
    {
        "workflow_preset": workflow_preset_source,
        "training_config_resource": training_config_name,
        "alpha_express": ALPHA_EXPRESS,
        "alpha_spatial": ALPHA_SPATIAL,
        "seed": SEED,
        "classifier_k": K_NEIGHBORS,
        "interaction_cutoff": INTERACTION_CUTOFF,
        "edge_predictor_threshold": EDGE_PREDICTOR_THRESHOLD,
        "edge_predictor_path_for_new_training": EDGE_PREDICTOR_PATH,
        "formal_scope_enabled": RUN_FORMAL_SCOPE,
        "formal_observed_times": FORMAL_OBSERVED_TIMES,
        "formal_interpolated_times": FORMAL_INTERPOLATED_TIMES,
        "formal_sde_n_samples": FORMAL_SDE_N_SAMPLES,
        "sde_dt": SDE_DT,
        "split_sde_dt": SPLIT_SDE_DT,
        "split_sigma": SPLIT_SIGMA,
        "split_growth_alpha": SPLIT_GROWTH_ALPHA,
    }
)
resolved_settings


## 3. Optional training

Training is off by default because a production run is long and should be launched
deliberately. The public `fit` call reads the complete stage plan from the wheel resource and the formal graph settings
from the workflow preset; this notebook does not reproduce the trainer in local cells.


In [ ]:
if RUN_TRAINING:
    cb.tl.fit(
        adata,
        config=config,
        device=DEVICE,
        ckpt_dir=MODEL_DIR,
        interaction_cutoff=INTERACTION_CUTOFF,
        edge_predictor_path=str(EDGE_PREDICTOR_PATH),
        edge_predictor_threshold=EDGE_PREDICTOR_THRESHOLD,
        evaluate_after_training=False,
    )
else:
    print("Training skipped. Set RUN_TRAINING=True only for a deliberate new run.")


## 4. Load an existing model

The model input concatenates aligned spatial coordinates followed by expression PCs.
The loader selects the configured dynamical and final score stages from `MODEL_DIR`.


In [ ]:
spatial_dim = int(adata.obsm[SPATIAL_KEY].shape[1])
latent_dim = int(adata.obsm[LATENT_KEY].shape[1])
model_dim = spatial_dim + latent_dim if CONCAT_SPATIAL else latent_dim

loaded = cb.tl.load_dynamical_model_from_dir(
    MODEL_DIR,
    dim=model_dim,
    device=DEVICE,
)
runtime = cb.tl.build_dynamical_runtime(loaded)

pd.Series(
    {
        "model_dimension": model_dim,
        "dynamical_stage": loaded.weight_stage,
        "score_stage": loaded.score_stage,
    }
)


## 5. Interpolate and assign cell types

With `RUN_FORMAL_SCOPE=False`, this walkthrough inserts one midpoint per observed
interval and caps particles for interactive use. `RUN_FORMAL_SCOPE=True` selects the
complete interpolation grid and particle policy from the preset. Classifier and SDE
settings always come from that same preset. The formal annotation policy uses `k=10`;
`k=1,5,20,50` remain labeled sensitivity settings.

No spatial warp is enabled. If you later enable one for a mosaic or video, keep
`spatial_warp_visualization_only=True` and use `communication_adata_dict` for every
numerical analysis.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
aligned_table, _ = cb.tl.adata_to_aligned_dataframe(
    adata,
    time_key=TIME_KEY,
    obsm_key=LATENT_KEY,
    spatial_key=SPATIAL_KEY,
    concat_spatial=CONCAT_SPATIAL,
    annotation_key=ANNOTATION_KEY,
)

compact_interpolated_times = [
    (left + right) / 2.0
    for left, right in zip(observed_times[:-1], observed_times[1:])
]
if RUN_FORMAL_SCOPE:
    interpolation_times = list(FORMAL_INTERPOLATED_TIMES)
    analysis_particles = (
        None if FORMAL_SDE_N_SAMPLES is None else int(FORMAL_SDE_N_SAMPLES)
    )
    scope_label = "formal preset"
else:
    interpolation_times = compact_interpolated_times
    analysis_particles = COMPACT_PARTICLES
    if FORMAL_SDE_N_SAMPLES is not None:
        analysis_particles = min(analysis_particles, int(FORMAL_SDE_N_SAMPLES))
    scope_label = "compact walkthrough"
analysis_times = sorted(set(observed_times + interpolation_times))
time_values = pd.to_numeric(adata.obs[TIME_KEY], errors="raise").to_numpy(float)
initial_population_size = int(np.isclose(time_values, observed_times[0]).sum())
evaluation_particles = (
    initial_population_size
    if analysis_particles is None
    else min(initial_population_size, int(analysis_particles))
)

pd.Series({
    "scope": scope_label,
    "analysis_times": analysis_times,
    "particle_cap": analysis_particles,
    "split_sde_dt": SPLIT_SDE_DT,
    "split_sigma": SPLIT_SIGMA,
    "split_growth_alpha": SPLIT_GROWTH_ALPHA,
})

trajectory = cb.tl.run_interpolation_workflow(
    df=aligned_table,
    dim=model_dim,
    annotation_key=ANNOTATION_KEY,
    runtime=runtime,
    device=DEVICE,
    output_dir=str(OUTPUT_DIR / "interpolation"),
    requested_plot_points=analysis_times,
    interp_time_points=interpolation_times,
    max_observed_timepoints=len(observed_times),
    use_real_for_observed=True,
    classifier_cache_path=str(OUTPUT_DIR / "classifier_resmlp.pt"),
    classifier_adata=adata,
    classifier_time_key=TIME_KEY,
    classifier_obsm_key=LATENT_KEY,
    classifier_spatial_key=SPATIAL_KEY,
    classifier_concat_spatial=CONCAT_SPATIAL,
    classifier_epochs=CLASSIFIER_EPOCHS,
    classifier_hidden_size=CLASSIFIER_HIDDEN_SIZE,
    classifier_lr=CLASSIFIER_LR,
    classifier_test_size=0.1,
    classifier_best_metric=CLASSIFIER_BEST_METRIC,
    classifier_strict_stratification=CLASSIFIER_STRICT_STRATIFICATION,
    classifier_knn_neighbors=K_NEIGHBORS,
    sde_n_samples=analysis_particles,
    skip_nonsplit_sde=not LINEAGE_ENABLED,
    sde_dt=SDE_DT,
    split_sde_dt=SPLIT_SDE_DT,
    split_sigma_scalar=SPLIT_SIGMA,
    split_growth_alpha=SPLIT_GROWTH_ALPHA,
    split_interaction_m=1024,
    spatial_warp_to_observed=False,
    random_seed=SEED,
)

pd.Series(
    {
        "time_grid": trajectory.ts_points,
        "interpolated_times": trajectory.interp_points,
        "classifier_accuracy": trajectory.classifier_accuracy,
        "classifier_balanced_accuracy": trajectory.classifier_balanced_accuracy,
        "knn_neighbors": K_NEIGHBORS,
        "scope": scope_label,
    }
)


## 6. Cell-type composition

Fractions are comparable across time. Counts depend on the requested particle cap,
so do not present them as population-size predictions unless the full population was
simulated under the intended growth/resampling contract.


In [ ]:
labels_by_time = [
    trajectory.adata_dict[key].obs[ANNOTATION_KEY].astype(str).to_numpy()
    for key in trajectory.time_keys
]
composition = cb.tl.summarize_label_composition(
    labels_by_time,
    trajectory.ts_points,
)
composition.head(10)


## 7. Velocity and growth

Velocity is recomputed separately within each real time slice. `reuse_if_present=False`
prevents stale arrays from replacing the current checkpoint calculation. Growth below
is evaluated on the unwarped state used for communication and LR analysis.


In [ ]:
velocity = cb.tl.compute_velocity_components_from_adata(
    adata,
    loaded.model,
    dim=model_dim,
    interaction_m=1024,
    interaction_threshold=INTERACTION_CUTOFF,
    device=DEVICE,
    time_key=TIME_KEY,
    obsm_key=LATENT_KEY,
    spatial_key=SPATIAL_KEY,
    concat_spatial=CONCAT_SPATIAL,
    write_to_adata=True,
    reuse_if_present=False,
)

velocity_summary = pd.DataFrame(
    {
        name: {
            "mean_norm": np.linalg.norm(values, axis=1).mean(),
            "median_norm": np.median(np.linalg.norm(values, axis=1)),
        }
        for name, values in velocity.items()
        if name in {"drift", "interaction", "score", "full"}
    }
).T
velocity_summary


In [ ]:
growth = cb.tl.evaluate_growth_by_timepoint(
    trajectory.communication_adata_dict,
    loaded.model,
    time_points=trajectory.ts_points,
    time_keys=trajectory.time_keys,
    annotation_key=ANNOTATION_KEY,
    spatial_key="spatial",
    device=DEVICE,
)
growth.groupby("time")["growth_rate"].agg(["mean", "median", "std"])


## 8. Sparse attention and cell-type communication

The package constructs only spatial neighbor edges and does not materialize an
all-cells-by-all-cells attention matrix. Communication is calculated from the
unwarped model state. Dense attention export remains off.


In [ ]:
communications = cb.tl.compute_timepoint_communications(
    adata_dict=trajectory.communication_adata_dict,
    time_points=trajectory.ts_points,
    annotation_key=ANNOTATION_KEY,
    f_net=runtime.f_net,
    device=DEVICE,
    out_dir=str(OUTPUT_DIR / "communication"),
    save_dense_attention_matrix=False,
    max_cells_per_timepoint=analysis_particles,
    random_seed=SEED,
)
list(communications)


## 9. Ligand-receptor trajectories

The main analysis requires every subunit of a ligand or receptor complex and uses
the least-expressed subunit (`complex_mode='min'`). Geometric mean is reserved for
a labeled sensitivity analysis. Generated log1p expression is reconstructed per cell
from the PCA state before cell-type means are calculated.


In [ ]:
lr_projection = cb.tl.project_communication_to_lr_timecourses(
    trajectory.communication_adata_dict,
    reference_adata=adata,
    communications=communications,
    lr_database=LR_DATABASE,
    time_points=trajectory.ts_points,
    annotation_key=ANNOTATION_KEY,
    matrix_key="M_per_source",
    spatial_dim=spatial_dim,
    expression_space="log1p",
    complex_mode="min",
    require_all_subunits=True,
    observed_adata=adata,
    observed_time_key=TIME_KEY,
    observed_time_points=observed_times,
    observed_annotation_key=ANNOTATION_KEY,
    observed_expression_space="log1p",
    return_type_matrices=False,
)
lr_projection.pair_timecourse.head(10)


## 10. Unwarped quantitative evaluation

This evaluator starts from the earliest observed population and reports joint,
physical-space, and PCA-space W1/W2 together with total-mass variation. It simulates
the native model state directly; display coordinates never enter these metrics.
Full-data checkpoints provide reconstruction evidence, not held-out forecasting.


In [ ]:
distribution_evaluation = cb.tl.evaluate_model_distributions(
    adata,
    loaded.model,
    time_points=observed_times,
    n_samples=evaluation_particles,
    dt=SDE_DT,
    sigma=SPLIT_SIGMA,
    include_score=True,
    interaction_m=1024,
    max_ot_points=1024,
    structure_max_points=5_000,
    device=DEVICE,
    time_key=TIME_KEY,
    obsm_key=LATENT_KEY,
    spatial_key=SPATIAL_KEY,
    concat_spatial=CONCAT_SPATIAL,
    random_seed=SEED,
    include_initial_time=False,
    verbose=True,
)
distribution_evaluation.metrics.groupby("space")[["w1", "w2", "tmv"]].mean()


## Reading the results

- Treat the classifier balanced accuracy as held-out model-selection performance, not
  an independent biological test set.
- Composition fractions should sum to one at each time; inspect rare types rather than
  relying only on the dominant class.
- Compare intrinsic drift, interaction, score, and full velocity norms, then inspect
  spatial direction fields before assigning a mechanism.
- Lower W1, W2, and TMV are better, but spatial, PCA, and joint spaces answer different
  questions. No single metric establishes universal superiority.
- Review `lr_projection.trajectory_coverage` and `dropped_trajectories` before interpreting
  top LR pairs; an unavailable subunit is not evidence of zero biology.

## Common pitfalls

- A cutoff such as 0.05 is not portable across coordinate systems. Use the cutoff and edge threshold saved in the wheel-packaged ARISTA formal preset.
- Keep `alpha_express=0.015`, `alpha_spatial=10`, and seed 42 fixed for the main run.
  Use `0.05` only as a clearly labeled sensitivity condition.
- Keep the dataset cutoff and edge threshold from its wheel-packaged formal preset; do not copy
  numbers between datasets.
- Keep quantitative analyses on unwarped states. Spatial warping is only for display.
- The formal classifier smoothing setting is `k=10`; `k=1,5,20,50` are sensitivity checks.


## Exercises

1. Add a second interpolation point to one interval and verify that observed anchors remain unchanged.
2. Re-label the same saved trajectory with `k = 1, 5, 10, 20, 50`; compare composition fractions
   and boundary-cell changes while keeping the formal `k=10` policy as the main result.
3. Re-run LR projection with `complex_mode='geometric_mean'` and
   `require_all_subunits=True`. Compare rankings against strict minimum without changing any
   other input.
4. For a genuine forecasting exercise, fit a new checkpoint with one observed target removed
   and state exactly which preprocessing steps did or did not see that target.


In [ ]:
# Exercise scaffold: keep the data, trajectory, and seed fixed.
K_SENSITIVITY = (1, 5, 10, 20, 50)
LR_COMPLEX_SENSITIVITY = ("min", "geometric_mean")

exercise_plan = pd.DataFrame(
    {
        "analysis": ["classifier smoothing", "LR complex aggregation"],
        "main_setting": ["formal spatial k=10", "min + all subunits"],
        "sensitivity": [str(K_SENSITIVITY), str(LR_COMPLEX_SENSITIVITY)],
    }
)
exercise_plan
